In [1]:
"""

Структура проекта:
  - ГОТОВО - обработка входящего формата (PDF, txt, ...), преобразование его в текстовую строку
  - ГОТОВО - Сентенизация, токенизация
  - ГОТОВО - NER
  - ГОТОВО - Сбор датасета
  - ГОТОВО - Модель BERT, цикл обучение+тестирование
  - TODO -   Интерфейс
  - TODO -   Рефакторинг для быстрой подгрузки уже обученной модели
"""

'\n\nСтруктура проекта:\n  - ГОТОВО - обработка входящего формата (PDF, txt, ...), преобразование его в текстовую строку\n  - ГОТОВО - Сентенизация, токенизация\n  - ГОТОВО - NER\n  - ГОТОВО - Сбор датасета\n  - ГОТОВО - Модель BERT, цикл обучение+тестирование\n  - TODO -   Интерфейс\n'

In [2]:
!pip install transformers -q
!pip install pubchempy
!pip install seqeval evaluate -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.3 MB/s eta 0:00:00


In [4]:
"""

Парсим данные с PubChem

"""

import os
import pubchempy as pcp
import pandas as pd

def get_chemical_names(limit=15000):
    print(f"Загрузка данных для {limit} соединений... (это может занять время)")
    # Запрашиваем CID (ID соединений) в диапазоне
    # Берём первые limit соединений
    chemical_list = []

    # Поиск по диапазону CID
    # PubChem позволяет получать свойства списком, что быстрее
    chunk_size = 1000
    for i in range(1, limit + 1, chunk_size): # разбиваем загрузку на чанки для удобного отслеживания прогресса
        cids = list(range(i, min(i + chunk_size, limit + 1)))
        try:
            # Получаем синонимы и/или IUPAC названия
            properties = pcp.get_properties('iupacname', cids)
            for prop in properties:
                if 'IUPACName' in prop:
                    chemical_list.append(prop['IUPACName'])
            print(f"Загружено: {len(chemical_list)}")
        except Exception as e:
            print(f"Ошибка на итерации {i}: {e}")
    return chemical_list



csv_filename = 'chemicals_list.csv'

df_chemicals = None
if os.path.exists(csv_filename):
  # Полгружаем данные для обучения
  df_chemicals = pd.read_csv(csv_filename)
else:
  # Запуск скрейпинга
  # запрос может занять несколько минут
  chemicals = get_chemical_names(15000)
  # Сохранение в DataFrame и файл
  df_chemicals = pd.DataFrame(chemicals, columns=['Chemical_Name'])
  df_chemicals.to_csv(csv_filename, index=False)
  print(f"Готово! Получено названий: {len(df_chemicals)}")

print(df_chemicals.head(10))

                                       Chemical_Name
0        3-acetyloxy-4-(trimethylazaniumyl)butanoate
1     (2-acetyloxy-3-carboxypropyl)-trimethylazanium
2  5,6-dihydroxycyclohexa-1,3-diene-1-carboxylic ...
3                                 1-aminopropan-2-ol
4         (3-amino-2-oxopropyl) dihydrogen phosphate
5                        1-chloro-2,4-dinitrobenzene
6                               9-ethylpurin-6-amine
7               2,3-dihydroxy-3-methylpentanoic acid
8  (2,3,4,5,6-pentahydroxycyclohexyl) dihydrogen ...
9  2-[[4-[(2-amino-4-oxo-5,6,7,8-tetrahydro-1H-pt...


In [5]:
from transformers import pipeline
from torch.utils.data import Dataset
import torch

# Класс датасета для эффективной передачи данных в пайплайн
class ChemicalDataset(Dataset):
    def __init__(self, chemical_list, count):
        self.data = chemical_list[:count]
    def __len__(self):
        return len(self.data)
    def __getitem__(self, i):
        return f"Write a scientific or descriptive sentence about the chemical compound: {self.data[i]}.\nSentence:"

"""
Генерация с помощью GPT-2
Пока закомментирована, чтобы не конфликтовать с моделью TinyLlama
"""
# model_name = 'gpt2'

# # Инициализация пайплайна
# device = 0 if torch.cuda.is_available() else -1
# generator = pipeline('text-generation', model=model_name, device=device)
# # установка pad_token_id и padding_side для корректного батчинга
# generator.tokenizer.pad_token_id = generator.model.config.eos_token_id
# generator.tokenizer.padding_side = 'left'

# def generate_sentences_optimized(chemical_list, count=500, batch_size=16):
#     dataset = ChemicalDataset(chemical_list, count)
#     results = []
#     # проходим по пайплайну как по итератору с поддержкой batch_size
#     for i, output in enumerate(generator(dataset, batch_size=batch_size, max_new_tokens=30, truncation=True)):
#         chemical = chemical_list[i]
#         prompt = f"Write a scientific or descriptive sentence about the chemical compound: {chemical}.\nSentence:"
#         generated_text = output[0]['generated_text']
#         sentence = generated_text.replace(prompt, "").strip().split('\n')[0]
#         results.append({"Chemical": chemical, "Sentence": sentence})
#     return pd.DataFrame(results)
# sentences_df = generate_sentences_optimized(chemicals, count=500, batch_size=16)
# print(sentences_df.head())

'\nГенерация с помощью GPT-2\nПока закомментирована, чтобы не конфликтовать с моделью TinyLlama\n'

In [6]:
"""
Генерация с помощью TinyLlama
"""

def load_generative_model(model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0"):
  device = 0 if torch.cuda.is_available() else -1
  # Инициализация нового пайплайна
  new_generator = pipeline(
      'text-generation',
      model=model_name,
      device=device,
      dtype=torch.float16 if torch.cuda.is_available() else torch.float32
  )
  # Установка параметров токенизатора
  new_generator.tokenizer.pad_token_id = new_generator.tokenizer.eos_token_id
  new_generator.tokenizer.padding_side = 'left'
  print(f"Модель {model_name} готова к работе!")
  return new_generator

def generate_with_tinyllama(chemical_list, count=10, batch_size=8):
    dataset = ChemicalDataset(chemical_list, count)
    results = []
    print(f"Начинаем генерацию для {count} элементов с batch_size={batch_size}...")
    new_generator = load_generative_model()
    # для пакетной генерации пайплайну нужен доступ к датасету и указание batch_size
    for i, output in enumerate(new_generator(dataset, batch_size=batch_size, max_new_tokens=50, truncation=True)):
        chemical = chemical_list[i]
        generated_text = output[0]['generated_text']

        # Извлекаем только сгенерированное предложение после метки Sentence:
        if "Sentence:" in generated_text:
            sentence = generated_text.split("Sentence:")[-1].strip().split('\n')[0]
        else:
            sentence = generated_text.strip().split('\n')[-1]
        results.append({"Chemical": chemical, "Sentence": sentence})
        if (i + 1) % 500 == 0:
            print(f"Обработано: {i + 1}")

    return pd.DataFrame(results)

if os.path.exists('sentences.csv'):
  tinyllama_results = pd.read_csv('sentences.csv')
else:
  # Ставим большой batch_size для ускорения
  tinyllama_results = generate_with_tinyllama(chemicals, count=15000, batch_size=32) #пока генерим только ламой
  print(tinyllama_results.head())
  # Сохранение сгенерированных предложений
  tinyllama_results.to_csv('sentences.csv', index=False)

In [7]:
import spacy
import pandas as pd
nlp = spacy.load("en_core_web_sm")


def tokenize_and_label(df):
    labeled_data = []
    for _, row in df.iterrows():
        chemical = row['Chemical']
        sentence = row['Sentence']

        doc = nlp(sentence)
        # Помечаем токены
        tokens_with_labels = []
        for token in doc:
            token_text = token.text
            # Проверяем, является ли токен (включая знаки препинания) частью названия химиката
            is_chemical = False
            if token_text.lower() in chemical.lower():
                is_chemical = True
            tokens_with_labels.append((token_text, is_chemical))
        labeled_data.append({
            'Chemical': chemical,
            'Tokens_and_Labels': tokens_with_labels
        })

    return pd.DataFrame(labeled_data)

# Применяем к результатам от LLM
final_tagged_df = None
if os.path.exists('final_tagged_chemicals.csv'):
  final_tagged_df = pd.read_csv('final_tagged_chemicals.csv')
else:
  final_tagged_df = tokenize_and_label(tinyllama_results)
  # Вывод примера
  if not final_tagged_df.empty:
      print(f"Chemical: {final_tagged_df.iloc[0]['Chemical']}")
      print(final_tagged_df.iloc[0]['Tokens_and_Labels'])
      # Сохранение итогового размеченного датасета в CSV
  final_tagged_df.to_csv('final_tagged_chemicals.csv', index=False)
  display(final_tagged_df.head())

In [8]:
"""

Размечаем датасет в BIO-разметке

"""
def convert_to_bio(tokens_with_labels):
    bio_labels = []
    prev_is_chemical = False

    for token, is_chemical in tokens_with_labels:
        if is_chemical:
            if not prev_is_chemical:
                bio_labels.append("B-CHEM")
            else:
                bio_labels.append("I-CHEM")
            prev_is_chemical = True
        else:
            bio_labels.append("O")
            prev_is_chemical = False
    return bio_labels


final_tagged_df = None
if os.path.exists("final_bio_tagged_chemicals.csv"):
  final_tagged_df = pd.read_csv("final_bio_tagged_chemicals.csv")
else:
  # Применяем конвертацию к датафрейму
  final_tagged_df['OIB_Labels'] = final_tagged_df['Tokens_and_Labels'].apply(convert_to_bio)
  # Посмотрим на результат для первой строки
  print(f"Chemical: {final_tagged_df.iloc[0]['Chemical']}")
  print("Tokens and BIO Labels:")
  example_tokens = [t[0] for t in final_tagged_df.iloc[0]['Tokens_and_Labels']]
  example_labels = final_tagged_df.iloc[0]['OIB_Labels']
  for t, l in zip(example_tokens, example_labels):
      print(f"{t:30} {l}")
  # Сохраняем обновленный датасет
  final_tagged_df.to_csv('final_bio_tagged_chemicals.csv', index=False)
  display(final_tagged_df.head())

In [9]:
"""

Предобработка текста завершена, начинается подготовка датасета для обучения
(разбиение на выборки, конвертация в нужный формат)

"""

from datasets import Dataset, DatasetDict, Features, Sequence, Value, ClassLabel
from sklearn.model_selection import train_test_split
import ast

# В CSV списки сохраняются как строки, преобразуем их обратно в списки Python
if isinstance(final_tagged_df['OIB_Labels'].iloc[0], str):
    final_tagged_df['OIB_Labels'] = final_tagged_df['OIB_Labels'].apply(ast.literal_eval)
if isinstance(final_tagged_df['Tokens_and_Labels'].iloc[0], str):
    final_tagged_df['Tokens_and_Labels'] = final_tagged_df['Tokens_and_Labels'].apply(ast.literal_eval)

# Извлекаем только токены (без булевых флагов)
final_tagged_df['tokens'] = final_tagged_df['Tokens_and_Labels'].apply(lambda x: [t[0] for t in x])

# Определяем соответствие меток и ID
label_list = ['O', 'B-CHEM', 'I-CHEM']
label2id = {label: i for i, label in enumerate(label_list)}
final_tagged_df['ner_tags'] = final_tagged_df['OIB_Labels'].apply(lambda x: [label2id[label] for label in x])

# Разделяем на train и validation
train_df, val_df = train_test_split(final_tagged_df[['tokens', 'ner_tags']], test_size=0.2, random_state=42)

# Определяем схему данных (Features)
features = Features({
    'tokens': Sequence(Value('string')),
    'ner_tags': Sequence(ClassLabel(names=label_list))
})

# Создаем Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True), features=features)
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True), features=features)

raw_datasets = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset
})

print("Датасет подготовлен!")
print(raw_datasets)
print("Пример данных:", raw_datasets['train'][0])

Датасет подготовлен!
DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 11992
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 2999
    })
})
Пример данных: {'tokens': ['Bis(2', '-', 'fluoroethyl', ')', 'carbonate', 'is', 'a', 'strong', 'oxidizing', 'agent', ',', 'useful', 'for', 'removing', 'sulfur', 'from', 'organic', 'compounds', '.'], 'ner_tags': [1, 2, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


In [10]:
from transformers import AutoTokenizer

# Инициализируем токенизатор BioBERT
model_checkpoint = 'dmis-lab/biobert-v1.1'
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

print(f"Токенизатор для {model_checkpoint} успешно загружен!")

# Тестовая проверка токенизатора
# test_tokens = raw_datasets['train'][0]['tokens']
# test_encoding = tokenizer(test_tokens, is_split_into_words=True)
# print(f"Пример токенизации (input_ids): {test_encoding['input_ids'][:10]}...")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Токенизатор для dmis-lab/biobert-v1.1 успешно загружен!


In [11]:
import numpy as np
import evaluate
from transformers import DataCollatorForTokenClassification, TrainingArguments, Trainer, AutoModelForTokenClassification

# Функция выравнивания меток после токенизации
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx])
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Применяем токенизацию ко всему датасету
tokenized_datasets = raw_datasets.map(tokenize_and_align_labels, batched=True)
# Метрики для оценки
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# Инициализация модели
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label={i: l for i, l in enumerate(label_list)},
    label2id={l: i for i, l in enumerate(label_list)}
)

# Настройка гиперпараметров обучения
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3, # по факту эпох больше т.к. данные прогоняются через модель по нескольку раз
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Запуск обучения
print("Начинается обучение")
trainer.train()

# Сохранение итоговой модели
trainer.save_model("./finetuned_biobert")
tokenizer.save_pretrained("./finetuned_biobert")

Map:   0%|          | 0/11992 [00:00<?, ? examples/s]

Map:   0%|          | 0/2999 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dmis-lab/biobert-v1.1
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./finetuned_biobert/tokenizer_config.json',
 './finetuned_biobert/tokenizer.json')

In [16]:
# Сохраняем модель для последующей быстрой загрузки (без необходимости заново обучать)
# ВНИМАНИЕ! Модель весит почти 350 Мб, чем пытаться залить её в колаб - наверное проще переобучить

import shutil
if os.path.exists('finetuned_biobert'):
    shutil.make_archive('finetuned_biobert', 'zip', 'finetuned_biobert')
else:
    print(f"Папка не найдена.")

In [12]:
import torch

def predict_chemicals(text, model, tokenizer, label_list):
    # Токенизация входного текста
    inputs = tokenizer(text, return_tensors="pt", truncation=True, is_split_into_words=False)
    # Перенос данных на то же устройство, что и модель
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Получение предсказаний
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    # Определение наиболее вероятных меток
    predictions = torch.argmax(outputs.logits, dim=2)
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    # Выравнивание меток с токенами (пропускаем специальные токены)
    results = []
    for token, pred_idx in zip(tokens, predictions[0].tolist()):
        if token not in tokenizer.all_special_tokens:
            label = label_list[pred_idx]
            results.append((token, label))

    return results

In [14]:
sample_text = "Dimethoxybenzoate is bad for your health."
# Запуск предсказания
prediction = predict_chemicals(sample_text, model, tokenizer, label_list)

# Форматированный вывод результатов
print(f"{'Token':20} : {'Label'}")
print("-" * 30)
for token, label in prediction:
    print(f"{token:20} : {label}")


"""
Если всё прошло корректно, модель пометит "Диметоксибензоат" как химикат (тег B-CHEM),
а все остальные слова проигнорирует (тег O)

"""

Token                : Label
------------------------------
Di                   : B-CHEM
##met                : B-CHEM
##ho                 : B-CHEM
##xy                 : B-CHEM
##ben                : B-CHEM
##zo                 : B-CHEM
##ate                : B-CHEM
is                   : O
bad                  : O
for                  : O
your                 : O
health               : O
.                    : O


In [20]:
"""
Чистка метаданных ноутбука, чтобы Гитхаб нормально его отображал
Источник: https://github.com/Ali-mohammadi-design/Google-Colab-Notebook-Cleaner

"""
import json
from google.colab import files
from pathlib import Path

# Step 1: Upload the broken notebook
uploaded = files.upload()
notebook_filename = next(iter(uploaded))  # get uploaded filename

# Step 2: Clean the notebook's metadata
def clean_metadata(notebook_path):
    path = Path(notebook_path)
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if "widgets" in data.get("metadata", {}):
        print("🧹 Removing metadata.widgets...")
        del data["metadata"]["widgets"]

    cleaned_path = path.with_name(path.stem + "_CLEAN.ipynb")
    with open(cleaned_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=1)

    print(f"✅ Cleaned notebook saved as: {cleaned_path}")
    return cleaned_path

cleaned_file = clean_metadata(notebook_filename)

# Step 3: Download the cleaned notebook
files.download(str(cleaned_file))

Saving ВКР основной код.ipynb to ВКР основной код.ipynb
🧹 Removing metadata.widgets...
✅ Cleaned notebook saved as: ВКР основной код_CLEAN.ipynb


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>